# DDT Analyzer
### Which r/ssbm users have the biggest effect on Daily Discussion Thread activity?

This notebook scrapes DDT posts and comments from r/ssbm, then uses statistical analysis to rank each user's effect on comment count — controlling for day of week, time trends, seasonal patterns, and community events.

**All code is embedded directly in this notebook — no need to clone the repo.**

**Just click Runtime > Run all (or Ctrl+F9) to run everything.**

---
## Step 1: Install Dependencies

In [ ]:
!pip install -q requests pandas numpy scipy scikit-learn statsmodels

---
## Step 2: Settings
Adjust these before running if you want different thresholds.

In [ ]:
#@title Analysis Settings { display-mode: "form" }

#@markdown **User filters** — who qualifies for the tier list:
MIN_DDTS = 5        #@param {type: "integer"}
MIN_COMMENTS = 10   #@param {type: "integer"}

#@markdown **Display:**
TOP_N = 50           #@param {type: "integer"}

#@markdown **Ridge regression:**
RIDGE_ALPHA = 10.0   #@param {type: "number"}

#@markdown **Scraping:**
MAX_SEARCH_PAGES = 20  #@param {type: "integer"}

#@markdown **Use synthetic test data instead of scraping Reddit:**
USE_TEST_DATA = False  #@param {type: "boolean"}

print(f'Settings: min_ddts={MIN_DDTS}, min_comments={MIN_COMMENTS}, top_n={TOP_N}, ridge_alpha={RIDGE_ALPHA}, max_pages={MAX_SEARCH_PAGES}, test_data={USE_TEST_DATA}')

---
## Step 3: Scraper Code
Defines all the functions for fetching DDT posts and comments from Reddit, plus a synthetic test data generator.

In [ ]:
import json
import os
import re
import time
import random
from datetime import datetime, timedelta, timezone

import numpy as np
import requests

DATA_DIR = "data"
POSTS_FILE = os.path.join(DATA_DIR, "ddt_posts.json")
COMMENTS_DIR = os.path.join(DATA_DIR, "comments")

HEADERS = {
    "User-Agent": "DDTAnalyzer/1.0 (research script; analyzing DDT comment patterns)"
}
REQUEST_DELAY = 1.5


def _get_json(url, params=None, max_retries=3):
    """Fetch JSON from Reddit with rate limiting and retries."""
    for attempt in range(max_retries):
        time.sleep(REQUEST_DELAY)
        try:
            resp = requests.get(url, headers=HEADERS, params=params, timeout=30)
            if resp.status_code == 429:
                wait = int(resp.headers.get("Retry-After", 60))
                print(f"  Rate limited, waiting {wait}s...")
                time.sleep(wait)
                continue
            resp.raise_for_status()
            return resp.json()
        except (requests.RequestException, json.JSONDecodeError) as e:
            if attempt < max_retries - 1:
                wait = 2 ** (attempt + 1)
                print(f"  Request error ({e}), retrying in {wait}s...")
                time.sleep(wait)
            else:
                print(f"  Failed after {max_retries} attempts: {e}")
                return None
    return None


def search_ddt_posts(max_pages=20):
    """Search r/ssbm for Daily Discussion Thread posts."""
    posts = []
    after = None
    url = "https://www.reddit.com/r/ssbm/search.json"

    for page in range(max_pages):
        params = {
            "q": 'title:"Daily Discussion Thread"',
            "restrict_sr": "on",
            "sort": "new",
            "t": "all",
            "limit": 100,
        }
        if after:
            params["after"] = after

        print(f"  Fetching search page {page + 1}...")
        data = _get_json(url, params=params)
        if not data or "data" not in data:
            print("  No more search results or error.")
            break

        children = data["data"].get("children", [])
        if not children:
            break

        for child in children:
            post = child["data"]
            title = post.get("title", "")
            if not re.search(r"Daily Discussion Thread", title, re.IGNORECASE):
                continue

            posts.append({
                "id": post["id"],
                "title": title,
                "created_utc": post["created_utc"],
                "num_comments": post["num_comments"],
                "score": post.get("score", 0),
                "permalink": post.get("permalink", ""),
                "author": post.get("author", "[deleted]"),
            })

        after = data["data"].get("after")
        if not after:
            print(f"  Reached end of search results after {page + 1} pages.")
            break

    seen = set()
    unique_posts = []
    for p in posts:
        if p["id"] not in seen:
            seen.add(p["id"])
            unique_posts.append(p)

    unique_posts.sort(key=lambda p: p["created_utc"])
    return unique_posts


def _extract_comments_from_tree(tree, comments_list):
    """Recursively extract comments from Reddit's comment tree structure."""
    if isinstance(tree, list):
        for item in tree:
            _extract_comments_from_tree(item, comments_list)
        return

    if isinstance(tree, dict):
        kind = tree.get("kind")
        data = tree.get("data", {})

        if kind == "Listing":
            for child in data.get("children", []):
                _extract_comments_from_tree(child, comments_list)

        elif kind == "t1":
            author = data.get("author", "[deleted]")
            if author not in ("[deleted]", "[removed]", "AutoModerator"):
                comments_list.append({
                    "author": author,
                    "created_utc": data.get("created_utc", 0),
                    "score": data.get("score", 0),
                    "body_length": len(data.get("body", "")),
                    "id": data.get("id", ""),
                })
            replies = data.get("replies")
            if replies and isinstance(replies, dict):
                _extract_comments_from_tree(replies, comments_list)

        elif kind == "more":
            more_ids = data.get("children", [])
            if more_ids:
                comments_list.append({"_more_ids": more_ids, "_parent_id": data.get("parent_id", "")})


def _fetch_more_comments(post_id, comment_ids):
    """Fetch 'more comments' that weren't included in the initial response."""
    url = "https://www.reddit.com/api/morechildren.json"
    all_comments = []

    for i in range(0, len(comment_ids), 100):
        batch = comment_ids[i:i + 100]
        params = {
            "api_type": "json",
            "link_id": f"t3_{post_id}",
            "children": ",".join(batch),
            "limit_children": False,
            "sort": "top",
        }
        data = _get_json(url, params=params)
        if not data:
            continue

        things = data.get("json", {}).get("data", {}).get("things", [])
        for thing in things:
            if thing.get("kind") == "t1":
                tdata = thing["data"]
                author = tdata.get("author", "[deleted]")
                if author not in ("[deleted]", "[removed]", "AutoModerator"):
                    all_comments.append({
                        "author": author,
                        "created_utc": tdata.get("created_utc", 0),
                        "score": tdata.get("score", 0),
                        "body_length": len(tdata.get("body", "")),
                        "id": tdata.get("id", ""),
                    })

    return all_comments


def fetch_comments_for_post(post_id):
    """Fetch all comments for a given post, handling 'load more' stubs."""
    url = f"https://www.reddit.com/r/ssbm/comments/{post_id}.json"
    params = {"limit": 500, "depth": 10, "sort": "top"}

    data = _get_json(url, params=params)
    if not data or not isinstance(data, list) or len(data) < 2:
        return []

    comments = []
    _extract_comments_from_tree(data[1], comments)

    real_comments = [c for c in comments if "author" in c]
    more_stubs = [c for c in comments if "_more_ids" in c]

    all_more_ids = []
    for stub in more_stubs:
        all_more_ids.extend(stub["_more_ids"])

    if all_more_ids:
        print(f"    Fetching {len(all_more_ids)} additional comments...")
        extra = _fetch_more_comments(post_id, all_more_ids)
        real_comments.extend(extra)

    return real_comments


def scrape_all(max_search_pages=20, force_refresh_posts=False, force_refresh_comments=False):
    """Main scraping function. Fetches posts and comments, caching results to disk."""
    os.makedirs(DATA_DIR, exist_ok=True)
    os.makedirs(COMMENTS_DIR, exist_ok=True)

    if not force_refresh_posts and os.path.exists(POSTS_FILE):
        print(f"Loading cached posts from {POSTS_FILE}")
        with open(POSTS_FILE, "r") as f:
            posts = json.load(f)
        print(f"  {len(posts)} posts loaded from cache.")
    else:
        print("Searching for DDT posts on r/ssbm...")
        posts = search_ddt_posts(max_pages=max_search_pages)
        with open(POSTS_FILE, "w") as f:
            json.dump(posts, f, indent=2)
        print(f"  Found and cached {len(posts)} DDT posts.")

    all_comments = {}
    total = len(posts)
    for i, post in enumerate(posts):
        pid = post["id"]
        comment_file = os.path.join(COMMENTS_DIR, f"{pid}.json")

        if not force_refresh_comments and os.path.exists(comment_file):
            with open(comment_file, "r") as f:
                all_comments[pid] = json.load(f)
        else:
            print(f"  [{i + 1}/{total}] Fetching comments for: {post['title'][:60]}...")
            comments = fetch_comments_for_post(pid)
            with open(comment_file, "w") as f:
                json.dump(comments, f, indent=2)
            all_comments[pid] = comments
            print(f"    Got {len(comments)} comments.")

    return posts, all_comments


def generate_test_data():
    """Generate realistic synthetic DDT data for testing without scraping."""
    os.makedirs(DATA_DIR, exist_ok=True)
    os.makedirs(COMMENTS_DIR, exist_ok=True)

    rng = np.random.default_rng(42)
    random.seed(42)

    start_date = datetime(2025, 4, 10, 16, 0, 0, tzinfo=timezone.utc)
    n_days = 365

    users = [
        ("MeleeGod42", 0.40, 25, 8),
        ("FoxMain2007", 0.35, 20, 12),
        ("WaveDashWizard", 0.55, 12, 6),
        ("ShineSpike_", 0.45, 10, 5),
        ("PM_ME_TECH", 0.50, 8, 4),
        ("Falco_Enjoyer", 0.60, 5, 3),
        ("TourneyTO_Steve", 0.40, 5, 4),
        ("SetCountBot", 0.70, 3, 2),
        ("MarfMain", 0.55, 4, 3),
        ("SlippiRanked", 0.50, 4, 3),
        ("CasualMelee", 0.65, 0, 2),
        ("PuffIsLame", 0.50, 0, 3),
        ("UCFDebater", 0.45, -1, 2),
        ("NetplayAndy", 0.60, 0, 2),
        ("LocalSceneFan", 0.40, 1, 2),
        ("TopPlayerWatcher", 0.55, 0, 1),
        ("SmashClipGuy", 0.35, 1, 2),
        ("TechChaseKing", 0.45, 0, 2),
        ("TierListDebater", 0.50, -5, 4),
        ("RulesetComplainer", 0.35, -4, 3),
        ("SaltyRunback", 0.40, -3, 2),
        ("IciesApologist", 0.30, -3, 2),
        ("TrollPoster9000", 0.25, -10, 6),
        ("copypasta_bot_v2", 0.20, -8, 3),
    ]

    for i in range(40):
        name = f"ssbm_fan_{i:03d}"
        show_rate = rng.uniform(0.05, 0.30)
        effect = rng.normal(0, 1.5)
        own_comments = rng.integers(1, 3)
        users.append((name, show_rate, effect, int(own_comments)))

    dow_effects = {0: 10, 1: 8, 2: 5, 3: 12, 4: 15, 5: -5, 6: -8}

    posts = []
    all_comments = {}
    base_activity = 80

    for day_offset in range(n_days):
        dt = start_date + timedelta(days=day_offset)
        dow = dt.weekday()
        month = dt.month

        if rng.random() < 0.05:
            if posts:
                prev_id = posts[-1]["id"]
                extra = int(rng.integers(15, 50))
                for j in range(extra):
                    all_comments[prev_id].append({
                        "author": f"lurker_{rng.integers(0, 500):04d}",
                        "created_utc": dt.timestamp() + rng.integers(0, 86400),
                        "score": int(rng.integers(-1, 10)),
                        "body_length": int(rng.integers(5, 200)),
                        "id": f"c_{prev_id}_extra_{j}",
                    })
                posts[-1]["num_comments"] = len(all_comments[prev_id])
                comment_file = os.path.join(COMMENTS_DIR, f"{prev_id}.json")
                with open(comment_file, "w") as f:
                    json.dump(all_comments[prev_id], f, indent=2)
            continue

        post_id = f"sim_{day_offset:04d}"
        expected = base_activity + dow_effects[dow]

        if month in (6, 7, 12, 1):
            expected += 10
        elif month in (8, 9):
            expected -= 5

        expected += -0.02 * day_offset

        event_spike = 0
        if rng.random() < 0.05:
            event_spike = rng.integers(20, 60)
            expected += event_spike

        present_users = []
        for name, show_rate, effect, avg_own in users:
            adjusted_rate = show_rate
            if event_spike > 0:
                adjusted_rate = min(0.95, show_rate + 0.2)
            if dow >= 5:
                adjusted_rate *= 0.8
            if rng.random() < adjusted_rate:
                present_users.append((name, effect, avg_own))
                expected += effect

        expected += rng.normal(0, 10)
        total_comments = max(5, int(expected))

        comments = []
        for name, effect, avg_own in present_users:
            n_comments = max(1, int(rng.poisson(avg_own)))
            for j in range(n_comments):
                comments.append({
                    "author": name,
                    "created_utc": dt.timestamp() + rng.integers(0, 86400),
                    "score": int(rng.integers(-2, 20)),
                    "body_length": int(rng.integers(10, 500)),
                    "id": f"c_{post_id}_{name}_{j}",
                })

        while len(comments) < total_comments:
            anon_name = f"lurker_{rng.integers(0, 500):04d}"
            comments.append({
                "author": anon_name,
                "created_utc": dt.timestamp() + rng.integers(0, 86400),
                "score": int(rng.integers(-1, 10)),
                "body_length": int(rng.integers(5, 200)),
                "id": f"c_{post_id}_anon_{len(comments)}",
            })

        title_date = dt.strftime("%B %d, %Y")
        posts.append({
            "id": post_id,
            "title": f"Daily Discussion Thread {title_date}",
            "created_utc": dt.timestamp(),
            "num_comments": len(comments),
            "score": int(rng.integers(5, 30)),
            "permalink": f"/r/ssbm/comments/{post_id}/daily_discussion_thread/",
            "author": "AutoModerator",
        })

        comment_file = os.path.join(COMMENTS_DIR, f"{post_id}.json")
        with open(comment_file, "w") as f:
            json.dump(comments, f, indent=2)
        all_comments[post_id] = comments

    with open(POSTS_FILE, "w") as f:
        json.dump(posts, f, indent=2)

    total = sum(len(c) for c in all_comments.values())
    print(f"Generated {len(posts)} synthetic DDTs with {total:,} total comments")


print("Scraper functions defined.")

---
## Step 4: Analysis Code
Defines the statistical analysis pipeline: baseline model, residual analysis, Ridge regression, and tier assignment.

In [ ]:
from collections import Counter

import pandas as pd
from scipy import stats
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm


def load_data():
    """Load scraped posts and comments from disk into a DataFrame."""
    posts_file = os.path.join(DATA_DIR, "ddt_posts.json")
    comments_dir = os.path.join(DATA_DIR, "comments")

    if not os.path.exists(posts_file):
        print("Error: No scraped data found. Run the scraping step first.")
        return None

    with open(posts_file) as f:
        posts = json.load(f)

    records = []
    for post in posts:
        pid = post["id"]
        comment_file = os.path.join(comments_dir, f"{pid}.json")
        if not os.path.exists(comment_file):
            continue

        with open(comment_file) as f:
            comments = json.load(f)

        dt = datetime.fromtimestamp(post["created_utc"], tz=timezone.utc)
        commenters = set()
        commenter_counts = Counter()
        for c in comments:
            author = c.get("author", "")
            if author:
                commenters.add(author)
                commenter_counts[author] += 1

        records.append({
            "post_id": pid,
            "title": post["title"],
            "date": dt.date(),
            "datetime": dt,
            "day_of_week": dt.strftime("%A"),
            "day_of_week_num": dt.weekday(),
            "month": dt.month,
            "year": dt.year,
            "total_comments": len(comments),
            "num_unique_commenters": len(commenters),
            "commenters": commenters,
            "commenter_counts": dict(commenter_counts),
        })

    df = pd.DataFrame(records)
    if df.empty:
        print("Error: No DDT data loaded.")
        return None

    df = df.sort_values("date").reset_index(drop=True)
    df["time_index"] = range(len(df))
    df["rolling_mean_14"] = (
        df["total_comments"]
        .rolling(window=14, min_periods=1, center=True)
        .mean()
    )

    dates = pd.to_datetime(df["date"])
    gaps = dates.diff(periods=-1).abs().dt.days
    df["days_active"] = gaps.fillna(1).astype(int).clip(lower=1)

    n_multiday = (df["days_active"] > 1).sum()
    if n_multiday > 0:
        print(f"  Note: {n_multiday} DDTs were left up for multiple days (controlled for in model)")

    print(f"Loaded {len(df)} DDTs spanning {df['date'].min()} to {df['date'].max()}")
    print(f"  Total comments: {df['total_comments'].sum():,}")
    print(f"  Mean comments/DDT: {df['total_comments'].mean():.1f}")
    print(f"  Median comments/DDT: {df['total_comments'].median():.1f}")

    return df


def build_confound_features(df):
    """Build confound feature matrix (day of week, time trend, month, rolling activity)."""
    features = {}

    for day in ["Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]:
        features[f"dow_{day}"] = (df["day_of_week"] == day).astype(float)

    t = df["time_index"].values.astype(float)
    t_norm = t / max(t.max(), 1)
    features["trend_linear"] = t_norm
    features["trend_quadratic"] = t_norm ** 2

    for m in range(2, 13):
        features[f"month_{m}"] = (df["month"] == m).astype(float)

    rolling_lag = df["total_comments"].rolling(window=7, min_periods=1).mean().shift(1)
    rolling_lag = rolling_lag.fillna(df["total_comments"].mean())
    features["rolling_activity_7d"] = rolling_lag.values

    features["days_active"] = df["days_active"].values.astype(float)

    X = pd.DataFrame(features, index=df.index)
    return X


def fit_baseline_model(df, X_confounds):
    """Fit OLS baseline model: total_comments ~ confounders. Returns residuals."""
    y = df["total_comments"].values.astype(float)
    X = sm.add_constant(X_confounds.values.astype(float))

    model = sm.OLS(y, X).fit()

    print(f"\n--- Baseline Model (confounders only) ---")
    print(f"  R² = {model.rsquared:.3f}  (confounders explain {model.rsquared * 100:.1f}% of variance)")
    print(f"  Adjusted R² = {model.rsquared_adj:.3f}")

    print(f"\n  Day-of-week effects (vs Monday):")
    dow_features = [c for c in X_confounds.columns if c.startswith("dow_")]
    for i, feat in enumerate(dow_features):
        coef_idx = i + 1
        day_name = feat.replace("dow_", "")
        coef = model.params[coef_idx]
        pval = model.pvalues[coef_idx]
        sig = "*" if pval < 0.05 else ""
        print(f"    {day_name:>12s}: {coef:+.1f} comments  (p={pval:.3f}) {sig}")

    residuals = model.resid
    return residuals, model


def analyze_user_effects_residual(df, residuals, min_ddts=5, min_comments=10):
    """Residual-based analysis: compare mean residual when user is present vs absent."""
    all_users = Counter()
    user_total_comments = Counter()
    for _, row in df.iterrows():
        for user, count in row["commenter_counts"].items():
            all_users[user] += 1
            user_total_comments[user] += count

    eligible_users = {
        u for u, n_ddts in all_users.items()
        if n_ddts >= min_ddts and user_total_comments[u] >= min_comments
    }
    print(f"\n--- Residual-Based User Effect Analysis ---")
    print(f"  {len(all_users)} total unique users")
    print(f"  {len(eligible_users)} users meet thresholds (>={min_ddts} DDTs, >={min_comments} comments)")

    results = []
    n_ddts = len(df)

    for user in eligible_users:
        present_mask = np.array([user in row["commenters"] for _, row in df.iterrows()])
        absent_mask = ~present_mask

        n_present = present_mask.sum()
        n_absent = absent_mask.sum()

        if n_present < 2 or n_absent < 2:
            continue

        resid_present = residuals[present_mask]
        resid_absent = residuals[absent_mask]

        mean_present = resid_present.mean()
        mean_absent = resid_absent.mean()
        effect = mean_present - mean_absent

        t_stat, p_value = stats.ttest_ind(resid_present, resid_absent, equal_var=False)

        pooled_std = np.sqrt(
            ((n_present - 1) * resid_present.std() ** 2 + (n_absent - 1) * resid_absent.std() ** 2)
            / (n_present + n_absent - 2)
        )
        cohens_d = effect / pooled_std if pooled_std > 0 else 0

        own_comments = np.mean([
            row["commenter_counts"].get(user, 0)
            for _, row in df.iterrows()
            if user in row["commenters"]
        ])

        results.append({
            "user": user,
            "n_ddts_present": int(n_present),
            "n_ddts_absent": int(n_absent),
            "presence_rate": n_present / n_ddts,
            "effect_residual": effect,
            "mean_resid_present": mean_present,
            "mean_resid_absent": mean_absent,
            "t_stat": t_stat,
            "p_value": p_value,
            "cohens_d": cohens_d,
            "avg_own_comments": own_comments,
            "total_comments": user_total_comments[user],
        })

    results_df = pd.DataFrame(results)

    if results_df.empty:
        print("  No eligible users found. Try lowering thresholds.")
        return results_df

    n_tests = len(results_df)
    results_df["p_value_corrected"] = np.minimum(results_df["p_value"] * n_tests, 1.0)
    results_df["significant"] = results_df["p_value_corrected"] < 0.05

    results_df = results_df.sort_values("effect_residual", ascending=False).reset_index(drop=True)
    return results_df


def analyze_user_effects_ridge(df, X_confounds, min_ddts=5, min_comments=10, alpha=10.0):
    """Multi-user Ridge regression controlling for confounders and user-user correlations."""
    all_users = Counter()
    user_total_comments = Counter()
    for _, row in df.iterrows():
        for user, count in row["commenter_counts"].items():
            all_users[user] += 1
            user_total_comments[user] += count

    eligible_users = sorted([
        u for u, n in all_users.items()
        if n >= min_ddts and user_total_comments[u] >= min_comments
    ])

    if not eligible_users:
        print("\n--- Ridge Regression ---")
        print("  No eligible users found.")
        return pd.DataFrame()

    user_features = {}
    for user in eligible_users:
        user_features[f"user_{user}"] = np.array([
            1.0 if user in row["commenters"] else 0.0
            for _, row in df.iterrows()
        ])

    X_users = pd.DataFrame(user_features, index=df.index)
    X_all = pd.concat([X_confounds, X_users], axis=1)
    y = df["total_comments"].values.astype(float)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_all.values)

    model = Ridge(alpha=alpha)
    model.fit(X_scaled, y)

    n_confounds = X_confounds.shape[1]
    coefs = model.coef_
    stds = scaler.scale_
    coefs_original = coefs / stds

    user_coefs = {}
    for i, user in enumerate(eligible_users):
        coef_idx = n_confounds + i
        user_coefs[user] = coefs_original[coef_idx]

    y_pred = model.predict(X_scaled)
    ss_res = np.sum((y - y_pred) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot

    print(f"\n--- Ridge Regression (alpha={alpha}) ---")
    print(f"  {len(eligible_users)} users included")
    print(f"  R² = {r2:.3f}  (confounders + users explain {r2 * 100:.1f}% of variance)")

    results = []
    for user in eligible_users:
        results.append({
            "user": user,
            "ridge_coef": user_coefs[user],
            "n_ddts": all_users[user],
            "total_comments": user_total_comments[user],
        })

    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values("ridge_coef", ascending=False).reset_index(drop=True)
    return results_df


def compute_combined_ranking(residual_df, ridge_df):
    """Combine residual-based and Ridge-based rankings via rank-averaging."""
    if residual_df.empty or ridge_df.empty:
        if not residual_df.empty:
            residual_df["combined_score"] = residual_df["effect_residual"]
            return residual_df
        elif not ridge_df.empty:
            ridge_df["combined_score"] = ridge_df["ridge_coef"]
            return ridge_df
        return pd.DataFrame()

    merged = residual_df.merge(ridge_df[["user", "ridge_coef"]], on="user", how="inner")
    if merged.empty:
        return residual_df

    merged["rank_residual"] = merged["effect_residual"].rank(pct=True)
    merged["rank_ridge"] = merged["ridge_coef"].rank(pct=True)
    merged["combined_score"] = (merged["rank_residual"] + merged["rank_ridge"]) / 2

    merged = merged.sort_values("combined_score", ascending=False).reset_index(drop=True)
    return merged


def assign_tiers(combined_df):
    """Assign tier labels based on combined score percentiles."""
    if combined_df.empty:
        return combined_df

    df = combined_df.copy()
    tier_specs = [
        ("S", 0.95),
        ("A", 0.80),
        ("B", 0.55),
        ("C", 0.30),
        ("D", 0.10),
        ("F", 0.00),
    ]

    tiers = []
    for _, row in df.iterrows():
        s = row["combined_score"]
        assigned = "F"
        for tier_name, threshold in tier_specs:
            if s >= threshold:
                assigned = tier_name
                break
        tiers.append(assigned)

    df["tier"] = tiers
    return df


def print_tier_list(tiered_df, top_n=None):
    """Print the tier list in a readable format."""
    if tiered_df.empty:
        print("\nNo users to display.")
        return

    print("\n" + "=" * 90)
    print("  DDT USER PRESENCE EFFECT TIER LIST")
    print("  (Effect = how much a user's presence changes comment count, controlling for confounders)")
    print("=" * 90)

    tier_order = ["S", "A", "B", "C", "D", "F"]
    tier_labels = {
        "S": "SUPERSTAR  - DDTs light up when they show up",
        "A": "MAJOR      - Noticeably boosts discussion",
        "B": "SOLID      - Positive contributor",
        "C": "NEUTRAL    - Average presence effect",
        "D": "MINOR      - Below-average effect",
        "F": "QUIET      - Minimal measured effect",
    }

    count = 0
    for tier in tier_order:
        tier_users = tiered_df[tiered_df["tier"] == tier]
        if tier_users.empty:
            continue

        print(f"\n{'─' * 90}")
        print(f"  TIER {tier}: {tier_labels.get(tier, '')}")
        print(f"{'─' * 90}")
        print(f"  {'User':<24s} {'Effect':>8s} {'Ridge':>8s} {'DDTs':>6s} "
              f"{'Pres%':>6s} {'AvgOwn':>7s} {'Sig':>5s}")
        print(f"  {'─' * 22}   {'─' * 6}   {'─' * 6}   {'─' * 4}   "
              f"{'─' * 4}   {'─' * 5}   {'─' * 3}")

        for _, row in tier_users.iterrows():
            if top_n and count >= top_n:
                remaining = len(tiered_df) - count
                print(f"\n  ... and {remaining} more users (use --top-n 0 to show all)")
                return

            user = row["user"]
            if len(user) > 22:
                user = user[:20] + ".."

            effect = row.get("effect_residual", 0)
            ridge = row.get("ridge_coef", float("nan"))
            n_ddts = row.get("n_ddts_present", row.get("n_ddts", 0))
            pres = row.get("presence_rate", 0) * 100
            own = row.get("avg_own_comments", 0)
            sig = "***" if row.get("p_value_corrected", 1) < 0.001 else \
                  "**" if row.get("p_value_corrected", 1) < 0.01 else \
                  "*" if row.get("p_value_corrected", 1) < 0.05 else ""

            ridge_str = f"{ridge:+.1f}" if not np.isnan(ridge) else "N/A"

            print(f"  {user:<24s} {effect:+8.1f} {ridge_str:>8s} {n_ddts:6.0f} "
                  f"{pres:5.1f}% {own:7.1f} {sig:>5s}")
            count += 1

    print(f"\n{'=' * 90}")
    print("  Significance: * p<0.05  ** p<0.01  *** p<0.001  (Bonferroni-corrected)")
    print("  Effect: adjusted difference in total comments when user is present vs absent")
    print("  Ridge: marginal effect controlling for all other users simultaneously")
    print("  AvgOwn: user's own average comments per DDT they appear in")
    print(f"{'=' * 90}")


def print_summary_stats(df):
    """Print descriptive statistics about the DDT dataset."""
    print(f"\n{'=' * 60}")
    print("  DATASET SUMMARY")
    print(f"{'=' * 60}")
    print(f"  Date range: {df['date'].min()} to {df['date'].max()}")
    print(f"  Number of DDTs: {len(df)}")
    print(f"  Total comments: {df['total_comments'].sum():,}")

    print(f"\n  Comments per DDT:")
    print(f"    Mean:   {df['total_comments'].mean():.1f}")
    print(f"    Median: {df['total_comments'].median():.1f}")
    print(f"    Std:    {df['total_comments'].std():.1f}")
    print(f"    Min:    {df['total_comments'].min()}")
    print(f"    Max:    {df['total_comments'].max()}")

    print(f"\n  Comments by day of week:")
    dow_stats = df.groupby("day_of_week")["total_comments"].agg(["mean", "median", "count"])
    dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    for day in dow_order:
        if day in dow_stats.index:
            row = dow_stats.loc[day]
            print(f"    {day:>12s}: mean={row['mean']:.1f}  median={row['median']:.1f}  (n={row['count']:.0f})")

    multiday = df[df["days_active"] > 1]
    if not multiday.empty:
        print(f"\n  Multi-day DDTs: {len(multiday)}")
        for _, row in multiday.iterrows():
            print(f"    {row['date']} ({row['days_active']} days up, {row['total_comments']} comments)")
    else:
        print(f"\n  Multi-day DDTs: 0")

    all_users = set()
    for _, row in df.iterrows():
        all_users.update(row["commenters"])
    print(f"\n  Unique commenters: {len(all_users):,}")
    print(f"{'=' * 60}")


def run_analysis(min_ddts=5, min_comments=10, ridge_alpha=10.0, top_n=50):
    """Run the full analysis pipeline."""
    df = load_data()
    if df is None:
        return None

    print_summary_stats(df)

    X_confounds = build_confound_features(df)
    residuals, baseline_model = fit_baseline_model(df, X_confounds)

    residual_df = analyze_user_effects_residual(df, residuals, min_ddts=min_ddts, min_comments=min_comments)
    ridge_df = analyze_user_effects_ridge(df, X_confounds, min_ddts=min_ddts, min_comments=min_comments, alpha=ridge_alpha)

    combined = compute_combined_ranking(residual_df, ridge_df)
    tiered = assign_tiers(combined)

    display_n = top_n if top_n > 0 else None
    print_tier_list(tiered, top_n=display_n)

    output_file = os.path.join(DATA_DIR, "tier_list.csv")
    if not tiered.empty:
        tiered.to_csv(output_file, index=False)
        print(f"\nResults saved to {output_file}")

    return tiered


print("Analysis functions defined.")

---
## Step 5: Get Data
Either scrape real DDT data from Reddit (~30-45 min due to rate limits) or generate synthetic test data instantly.

Toggle `USE_TEST_DATA` in the Settings cell above to switch modes.

In [ ]:
if USE_TEST_DATA:
    print("Generating synthetic test data...")
    generate_test_data()
else:
    print("Scraping DDT data from Reddit...")
    posts, comments = scrape_all(max_search_pages=MAX_SEARCH_PAGES)
    total = sum(len(c) for c in comments.values())
    print(f'\nDone! {len(posts)} DDTs, {total:,} comments scraped.')

---
## Step 6: Run Analysis & Generate Tier List
This controls for confounders (day of week, time trends, seasonality, community-wide surges) and ranks every user.

In [ ]:
tiered_df = run_analysis(
    min_ddts=MIN_DDTS,
    min_comments=MIN_COMMENTS,
    ridge_alpha=RIDGE_ALPHA,
    top_n=TOP_N,
)

---
## Step 7: Explore Results
The full results are in a DataFrame you can filter, sort, and search.

In [ ]:
# Show all S and A tier users
if tiered_df is not None and not tiered_df.empty:
    top_tiers = tiered_df[tiered_df['tier'].isin(['S', 'A'])]
    print(f'S and A tier users: {len(top_tiers)}')
    display(top_tiers[['user', 'tier', 'effect_residual', 'ridge_coef', 'n_ddts_present', 'presence_rate', 'avg_own_comments', 'p_value_corrected']])
else:
    print('No results to display.')

In [ ]:
# Search for a specific user (change the name below)
search_user = 'example_username'  #@param {type: "string"}

match = tiered_df[tiered_df['user'].str.contains(search_user, case=False, na=False)]
if match.empty:
    print(f'No user matching "{search_user}" found in the tier list.')
    print(f'(They may not meet the minimum thresholds: {MIN_DDTS} DDTs, {MIN_COMMENTS} comments)')
else:
    print(match[['user', 'tier', 'effect_residual', 'ridge_coef', 'n_ddts_present', 'presence_rate', 'avg_own_comments']].to_string(index=False))

In [ ]:
# Download the full tier list as CSV
try:
    from google.colab import files
    tiered_df.to_csv('ddt_tier_list.csv', index=False)
    files.download('ddt_tier_list.csv')
    print('Downloading ddt_tier_list.csv...')
except ImportError:
    tiered_df.to_csv('ddt_tier_list.csv', index=False)
    print('Saved to ddt_tier_list.csv')